In [1]:
from pathlib import Path
from datetime import datetime, timezone
from typing import List, Dict, Optional
import json, uuid, getpass

import pandas as pd
import ipywidgets as widgets
from IPython.display import display, Markdown, clear_output

root = Path.cwd() / 'agents026'
data_dir = root / 'data'
exec_dir = data_dir / 'execution'
audit_dir = data_dir / 'audit'
audit_dir.mkdir(parents=True, exist_ok=True)

AUDIT_LOG_PATH = audit_dir / 'audit_log.jsonl'
OPERATOR = getpass.getuser()  # 'root' here; in production: SSO identity

print('HITL module paths ready. Audit log:', AUDIT_LOG_PATH)
print('Operator identity:', OPERATOR)

HITL module paths ready. Audit log: /workspace/shared/agents026/data/audit/audit_log.jsonl
Operator identity: root


In [2]:
def audit_event(event_type: str, incident_id: str, payload: Dict) -> Dict:
    """Append an immutable audit record. Every agent/human decision goes through here."""
    record = {
        'audit_id': f'aud-{uuid.uuid4().hex[:12]}',
        'timestamp_utc': datetime.now(timezone.utc).isoformat(),
        'event_type': event_type,          # e.g. action_pending_approval | human_decision | action_executed | action_rejected
        'incident_id': incident_id,
        'operator': OPERATOR,
        'payload': payload,
    }
    with open(AUDIT_LOG_PATH, 'a', encoding='utf-8') as f:
        f.write(json.dumps(record, default=str) + '\n')
    return record

print('Audit logger ready (append-only JSONL)')

Audit logger ready (append-only JSONL)


In [3]:
def mock_rollback_deploy(service, version=None):
    return {'status': 'executed', 'message': f'Simulated rollback of {service} to {version or "previous stable"}'}

def mock_rollback_config(service, version=None):
    return {'status': 'executed', 'message': f'Simulated config rollback for {service}'}

def mock_disable_feature_flag(service, flag_name='unknown_flag'):
    return {'status': 'executed', 'message': f'Simulated disabling feature flag {flag_name} on {service}'}

def mock_restart_service(service, **kw):
    return {'status': 'executed', 'message': f'Simulated restart of {service}'}

def mock_scale_service(service, replicas_delta=1, **kw):
    return {'status': 'executed', 'message': f'Simulated scale of {service} by +{replicas_delta}'}

def mock_drain_node(service, **kw):
    return {'status': 'executed', 'message': f'Simulated node drain for {service}'}

EXECUTORS = {
    'rollback_deploy': lambda a: mock_rollback_deploy(a['target'], a.get('parameters', {}).get('version')),
    'rollback_config': lambda a: mock_rollback_config(a['target'], a.get('parameters', {}).get('version')),
    'disable_feature_flag': lambda a: mock_disable_feature_flag(a['target'], str(a.get('parameters', {}).get('feature_flag', a.get('parameters', {}).get('flag_name', 'unknown_flag')))),
    'restart_service': lambda a: mock_restart_service(a['target']),
    'scale_service': lambda a: mock_scale_service(a['target']),
    'drain_node': lambda a: mock_drain_node(a['target']),
}

print('Mock infra executors ready:', list(EXECUTORS.keys()))

Mock infra executors ready: ['rollback_deploy', 'rollback_config', 'disable_feature_flag', 'restart_service', 'scale_service', 'drain_node']


In [4]:
INCIDENT_ID = 'inc-014'   # change to review another incident's queue

workflow_path = exec_dir / f'{INCIDENT_ID}_workflow.json'
with open(workflow_path, 'r', encoding='utf-8') as f:
    bundle = json.load(f)

pending_actions = [a for a in bundle['execution_result']['skipped_actions'] if a['status'] == 'skipped']

display(Markdown(f"### Approval queue for `{INCIDENT_ID}`\n**Root cause:** {bundle['rca_result']['root_cause_hypothesis']}\n\n**{len(pending_actions)} action(s) awaiting human approval:**"))
display(pd.DataFrame(pending_actions)[['action_id', 'action_type', 'target', 'expected_impact']])

# Log that these actions entered the approval queue
for a in pending_actions:
    audit_event('action_pending_approval', INCIDENT_ID, {
        'action_id': a['action_id'], 'action_type': a['action_type'],
        'target': a['target'], 'rationale': a.get('rationale'),
    })

decisions: Dict[str, Dict] = {}   # action_id -> decision record
print(f'\n{len(pending_actions)} pending decisions logged to audit trail')

### Approval queue for `inc-014`
**Root cause:** The root cause is a recent deployment that introduced a configuration change affecting the connection pool, leading to increased latency and error rates.

**2 action(s) awaiting human approval:**

,action_id,action_type,target,expected_impact
0,act-b0a01d98,rollback_deploy,payments-service,Potential downtime and service disruption duri...
1,act-2c22cfc8,disable_feature_flag,payments-service,Possible loss of performance improvements but ...



2 pending decisions logged to audit trail


In [5]:
def build_approval_panel(actions: List[Dict]):
    panels = []
    for action in actions:
        aid = action['action_id']
        header = widgets.HTML(
            f"<b>{action['action_type']}</b> → <code>{action['target']}</code> "
            f"&nbsp;<span style='color:#888'>({aid})</span><br>"
            f"<i>{action['description']}</i><br>"
            f"<b>Expected impact:</b> {action.get('expected_impact','-')}<br>"
            f"<b>Agent rationale:</b> {action.get('rationale','-')}"
        )
        approve_btn = widgets.Button(description='✅ Approve', button_style='success')
        reject_btn = widgets.Button(description='❌ Reject', button_style='danger')
        status_lbl = widgets.HTML("<b style='color:#b58900'>⏳ PENDING HUMAN DECISION</b>")
        out = widgets.Output()

        def make_handler(action=action, approve=None, status_lbl=status_lbl,
                         approve_btn=approve_btn, reject_btn=reject_btn, out=out):
            def handler(btn):
                decision = 'approved' if btn is approve_btn else 'rejected'
                aid = action['action_id']

                # 1. Record the human decision (audit)
                audit_event('human_decision', INCIDENT_ID, {
                    'action_id': aid, 'action_type': action['action_type'],
                    'target': action['target'], 'decision': decision,
                })

                # 2. Execute if approved
                with out:
                    clear_output()
                    if decision == 'approved':
                        executor = EXECUTORS.get(action['action_type'])
                        result = executor(action) if executor else {'status': 'failed', 'message': 'no executor'}
                        action['status'] = result['status']
                        audit_event('action_executed', INCIDENT_ID, {
                            'action_id': aid, 'action_type': action['action_type'],
                            'target': action['target'], 'result': result,
                        })
                        status_lbl.value = f"<b style='color:#2aa198'>✅ APPROVED & EXECUTED</b> — {result['message']}"
                    else:
                        action['status'] = 'rejected'
                        audit_event('action_rejected', INCIDENT_ID, {
                            'action_id': aid, 'action_type': action['action_type'],
                            'target': action['target'],
                        })
                        status_lbl.value = "<b style='color:#dc322f'>❌ REJECTED by operator</b> — action will not run"

                decisions[aid] = {'decision': decision, 'status': action['status']}
                approve_btn.disabled = True
                reject_btn.disabled = True
            return handler

        h = make_handler()
        approve_btn.on_click(h)
        reject_btn.on_click(h)

        panels.append(widgets.VBox(
            [header, widgets.HBox([approve_btn, reject_btn, status_lbl]), out],
            layout=widgets.Layout(border='1px solid #ccc', padding='10px', margin='6px 0')
        ))
    return widgets.VBox(panels)

display(Markdown('## 🛡️ Human-in-the-Loop Approval Gate'))
display(build_approval_panel(pending_actions))

## 🛡️ Human-in-the-Loop Approval Gate

In [6]:
# Update workflow file with post-approval statuses
approved_now = [a for a in pending_actions if a['status'] == 'executed']
rejected_now = [a for a in pending_actions if a['status'] == 'rejected']
still_pending = [a for a in pending_actions if a['status'] == 'skipped']

bundle['execution_result']['executed_actions'].extend(approved_now)
bundle['execution_result']['skipped_actions'] = rejected_now + still_pending
bundle['execution_result']['notes'].append(
    f'HITL gate processed at {datetime.now(timezone.utc).isoformat()}: '
    f'{len(approved_now)} approved, {len(rejected_now)} rejected, {len(still_pending)} undecided'
)

with open(workflow_path, 'w', encoding='utf-8') as f:
    json.dump(bundle, f, default=str, indent=2)

print(f'Workflow updated: {len(approved_now)} approved+executed, {len(rejected_now)} rejected, {len(still_pending)} undecided\n')

# Display the compliance audit trail (UC-6)
audit_df = pd.DataFrame([json.loads(line) for line in open(AUDIT_LOG_PATH, encoding='utf-8')])
display(Markdown('### 📋 Compliance Audit Trail (append-only)'))
display(audit_df[['timestamp_utc', 'event_type', 'incident_id', 'operator',
                  ]].assign(action=[p.get('action_id','') for p in audit_df['payload']],
                            detail=[p.get('decision', p.get('result',{}).get('message','') if isinstance(p.get('result'),dict) else '') for p in audit_df['payload']]))

Workflow updated: 1 approved+executed, 0 rejected, 1 undecided



### 📋 Compliance Audit Trail (append-only)

,timestamp_utc,event_type,incident_id,operator,action,detail
0,2026-06-11T18:07:47.576356+00:00,action_pending_approval,inc-014,root,act-b0a01d98,
1,2026-06-11T18:07:47.585018+00:00,action_pending_approval,inc-014,root,act-2c22cfc8,
2,2026-06-11T18:07:54.902593+00:00,human_decision,inc-014,root,act-2c22cfc8,approved
3,2026-06-11T18:07:54.907503+00:00,action_executed,inc-014,root,act-2c22cfc8,Simulated disabling feature flag connection_po...
